In [0]:
from pyspark.sql import functions as F

REF = "/Volumes/voebem/bronze/arquivos/referencia"
SEM_ASPAS = chr(0)


In [0]:
aerodromos = (
    spark.read.format("csv")
    .option("header", True)
    .option("sep", ";")
    .option("skipRows", 1)
    .option("quote", SEM_ASPAS) # desliga o quoting: aspas aqui são "segundos"
    .option("encoding", "ISO-8859-1")
    .option("inferSchema", True)
    .load(f"{REF}/AerodromosPublicos.csv")
)
aerodromos = aerodromos.select(
    F.col("Código OACI").alias("codigo_oaci"),
    F.col("CIAD").alias("ciad"),
    F.col("Nome").alias("nome"),
    F.col("Município").alias("municipio"),
    F.col("UF").alias("uf"),
    F.col("Município Servido").alias("municipio_servido"),
    F.col("UF Servido").alias("uf_servido"),
    F.col("LATGEOPOINT").alias("lat_geopoint"),
    F.col("LONGEOPOINT").alias("long_geopoint"),
    F.col("Latitude").alias("latitude"),
    F.col("Longitude").alias("longitude"),
    F.col("Altitude").alias("altitude"),
    F.col("Operação Diurna").alias("operacao_diurna"),
    F.col("Operação Noturna").alias("operacao_noturna"),
    F.col("Situação").alias("situacao"),
    F.col("Validade do Registro").alias("validade_registro"),
    F.col("Portaria de Registro").alias("portaria_registro"),
    F.col("Link Portaria").alias("link_portaria"),
).withColumn("_ingerido_em", F.current_timestamp())
aerodromos.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("voebem.bronze.aerodromos")

print(f"bronze.aerodromos: {spark.table('voebem.bronze.aerodromos').count():,} linhas")

display(spark.sql("SELECT codigo_oaci, nome, municipio, uf FROM voebem.bronze.aerodromos WHERE codigo_oaci IN ('SBRB', 'SBGR', 'SBSP', 'SBFZ')"))


bronze.aerodromos: 496 linhas


codigo_oaci,nome,municipio,uf
SBRB,Plácido de Castro,RIO BRANCO,Acre
SBFZ,Pinto Martins,FORTALEZA,Ceará
SBSP,São Paulo/Congonhas - Deputado Freitas Nobre,SÃO PAULO,São Paulo
SBGR,Guarulhos - Governador André Franco Montoro,GUARULHOS,São Paulo


In [0]:
def ler_empresas (arquivo: str):
    '''Lê um cadastro de empresas. Sem união, sem enriquecimento: uma tabela por arquivo.'''
    return (
        spark.read.format("csv")
        .option("header", True)
        .option("sep", ";")
        .option("skipRows", 1)
        .option("encoding", "UTF-8")
        .option("quote", '"')
        .load(f"{REF}/{arquivo}")
        .select(
            F.col("ICAO").alias("icao"),
            F.col("Estrangeira").alias("sigla_iata"),
            F.col("Razao").alias("razao_social"),
            F.col("Servico").alias("servico"),
            F.col("Endereco").alias("endereco"),
            F.col("Cidade").alias("cidade"),
            F.col("UF").alias("uf"),
            F.col("CEP").alias("cep"),
            F.col("Telefone").alias("telefone"),
            F.col("Email").alias("email"),
            F.col("Data").alias("data_registro"),
            F.col("Ativa").alias("situacao")

        )
        .withColumn("_ingerido_em", F.current_timestamp())
        .withColumn("_arquivo_origem", F.lit(arquivo))
    )

In [0]:
for arquivo, tabela in [
    ("pda_empresas_aereas_nacionais.csv", "voebem.bronze.empresas_aereas_nacionais"), ("pda_empresas_aereas_estrangeiros.csv", "voebem.bronze.empresas_aereas_estrangeiras"),
    ]:
        ler_empresas(arquivo).write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(tabela)
        print(f"{tabela}: {spark.table(tabela).count():,} linhas")
   

voebem.bronze.empresas_aereas_nacionais: 729 linhas
voebem.bronze.empresas_aereas_estrangeiras: 150 linhas


In [0]:
display(spark.sql("""
    SELECT 'empresas_aereas_nacionais' AS tabela, COUNT(*) AS linhas, COUNT (CASE WHEN icao IS NOT NULL AND icao <> '' THEN 1 END) AS com_icao FROM voebem.bronze.empresas_aereas_nacionais
    UNION ALL
    SELECT 'empresas_aereas_estrangeiras', COUNT(*), COUNT (CASE WHEN icao IS NOT NULL AND icao <> '' THEN 1 END) FROM voebem.bronze.empresas_aereas_estrangeiras
"""))

tabela,linhas,com_icao
empresas_aereas_nacionais,729,20
empresas_aereas_estrangeiras,150,149


In [0]:
display(spark.sql("""
    SELECT icao, razao_social, servico, uf, situacao
    FROM voebem.bronze.empresas_aereas_nacionais
    WHERE icao IN ('GLO', 'TAM', 'AZU', 'PAM')
    ORDER BY icao
"""))

display(spark.sql("""
    SELECT icao, razao_social, servico, situacao
    FROM voebem.bronze.empresas_aereas_estrangeiras
    WHERE icao IN ('ALL', 'TAP', 'AVA', 'ARG')
    ORDER BY icao
"""))

icao,razao_social,servico,uf,situacao
AZU,AZUL LINHAS AÉREAS BRASILEIRAS S/A,"TRANSPORTE AÉREO NÃO REGULAR, TRANSPORTE AÉREO REGULAR",SP,ATIVA
GLO,GOL LINHAS AÉREAS S.A. (EX- VRG LINHAS AÉREAS S.A.),"TRANSPORTE AÉREO NÃO REGULAR, TRANSPORTE AÉREO REGULAR",RJ,ATIVA
TAM,TAM LINHAS AÉREAS S.A.,TRANSPORTE AÉREO REGULAR,SP,ATIVA


icao,razao_social,servico,situacao
ARG,AEROLINEAS ARGENTINAS S/A,ESTRANGEIRA REGULAR,ATIVA
AVA,AEROVIAS DEL CONTINENTE AMERICANO S.A. AVIANCA,ESTRANGEIRA REGULAR,ATIVA
TAP,TAP - TRANSPORTES AÉREOS PORTUGUESES S/A,ESTRANGEIRA REGULAR,ATIVA


In [0]:
CODIGOS = [
    ("codigo_di", "0", "Etapa Regular"),
    ("codigo_di", "2", "Etapa Extra"),
    ("codigo_di", "3", "Etapa de Retorno"),
    ("codigo_di", "4", "Inclusão de Etapa"),
    ("codigo_di", "6", "Etapa Não Remunerada Sem Transporte de Objetos"),
    ("codigo_di", "7", "Etapa de Voo de Fretamento"),
    ("codigo_di", "9", "Etapa de Voo Charter"),
    ("codigo_di", "D", "Etapa de Voo Duplicada"),
    ("codigo_di", "E", "Etapa Não Remunerada Com Transporte de Objetos"),
    ("codigo_tipo_linha", "N", "Doméstica Mista"),
    ("codigo_tipo_linha", "C", "Doméstica Cargueira"),
    ("codigo_tipo_linha", "I", "Internacional Mista"),
    ("codigo_tipo_linha", "G", "Internacional Cargueira")
]

codigos = spark.createDataFrame(CODIGOS, "dominio string, codigo string, descricao string")

codigos.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("voebem.bronze.codigos_operacao")

print(f"bronze.codigos_operacao: {spark.table('voebem.bronze.codigos_operacao').count():,} linhas")

display(spark.table("voebem.bronze.codigos_operacao"))

bronze.codigos_operacao: 13 linhas


dominio,codigo,descricao
codigo_di,0,Etapa Regular
codigo_di,2,Etapa Extra
codigo_di,3,Etapa de Retorno
codigo_di,4,Inclusão de Etapa
codigo_di,6,Etapa Não Remunerada Sem Transporte de Objetos
codigo_di,7,Etapa de Voo de Fretamento
codigo_di,9,Etapa de Voo Charter
codigo_di,D,Etapa de Voo Duplicada
codigo_di,E,Etapa Não Remunerada Com Transporte de Objetos
codigo_tipo_linha,N,Doméstica Mista


In [0]:
display(spark.sql("SHOW TABLES IN voebem.bronze"))

database,tableName,isTemporary
bronze,aerodromos,false
bronze,anac_ingestao_controle,false
bronze,codigos_operacao,false
bronze,empresas_aereas_estrangeiras,false
bronze,empresas_aereas_nacionais,false
bronze,vra,false
